Parameters define what inputs a function accepts when called. Python gives rick flexibility in how argumetns are passed

which FastAPI heavily relies on map URL query parameters, path variables, and optional headers directly into our endpoint funcitons

## Positional, Keyword, and Default Parameters

- Positional arguments are matched by their order
- Keyword arguments are matched by parameter name (order doesn't matter, and readability improves)
- Default parameters provide a fallback value if the caller omits the argument 
    - they must always be placed after non-default parameters

In [1]:
def create_user(username: str, role: str = "viewer", is_active: bool = True) -> dict[str, str | bool]:
    return {"username" : username, "role": role, "is_active" : is_active}


# Positional + relying on defaults
print(create_user("devanath"))

print()

#Keyword arguments (order can be swapped)
print(create_user(is_active=False, username="alex", role="admin"))

{'username': 'devanath', 'role': 'viewer', 'is_active': True}

{'username': 'alex', 'role': 'admin', 'is_active': False}


## The mutable Default Argument Trap (Critical Bug to avoid)

Never use a mutable object like [] or {} as a default parameter value.

Python evaluates default arguments only once when the function is defined - not on every call 
 - meaning every call shares and mutates the exact same list! Always default to None instead

In [2]:
# WRONG  - 'tags' list is shared across every single function call!

def add_tag_buggy(tag: str, tags: list[str] = []) -> list[str]:
    tags.append(tag)
    return tags

print(add_tag_buggy("python")) 

# THe output will be ["python"] for the first function call 

print(add_tag_buggy("fastapi")) 

# This time the Output will be ["python","fastapi"] which is unexpected, and it happend because list is mutablbe

['python']
['python', 'fastapi']


In [3]:
# RIGHT : Use None as the default and create a fresh list inside

def add_tag_safe(tag: str, tags: list[str] | None = None) -> list[str]:
    if tags is None:
        tags = []
    tags.append(tag)
    return tags


print(add_tag_safe("python"))
print(add_tag_safe("fastapi"))

# Now the out put changes

['python']
['fastapi']


## Variable length arguments (*args and **kwargs)

- *args collects any extra positional arguments into a tuple
- **kwargs collects any extra keyword arguments into a dict

In [4]:
def log_event(event_type: str, *tags: str, **metadata: str | int) -> None:
    print(f"Event: {event_type}")
    print("Tags (tuple):", tags)
    print("Metadata (dict):", metadata)

log_event("USER_LOGIN", "auth", "security", user_id=43, ip="2.0.0.5")


Event: USER_LOGIN
Tags (tuple): ('auth', 'security')
Metadata (dict): {'user_id': 43, 'ip': '2.0.0.5'}


## Real - world Backend and API examples
In FastAPI endpoints and database query builders, default parameters handle optional pagination/filtering

while **kwargs dynamically forwards filter conditions to SQLAlchemy queires

In [5]:

# Example A - FastAPI styel Paginated Search Endpoint

def search_products(
    query: str,
    category: str | None = None,
    page: int = 1,
    limit: int = 20,
    sort_desc: bool = False,
) -> dict[str, str | int | bool | None]:
    offset = (page - 1) * limit

    return {
        "search": query,
        "category" : category,
        "offset" : offset,
        "limit": limit,
        "sort_desc": sort_desc,
    }

print(search_products("laptop", page=2, sort_desc=True))



{'search': 'laptop', 'category': None, 'offset': 20, 'limit': 20, 'sort_desc': True}


In [ ]:
# Example B  - Unpacking a dictionary with ** into Funciton parameters
query_params = {"query": "keyboard", "category": " accessories", "limit":10}

print(search_products(**query_params))


# This unpacks dict keys directly inot named parameters of the function

{'search': 'keyboard', 'category': ' accessories', 'offset': 0, 'limit': 10, 'sort_desc': False}
